# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

# Here it is - see the base_url

openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# message = "Hello, Llama! This is my first ever message to you! Hi!"



In [3]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [8]:
ed = Website("https://edwarddonner.com")
# ed.links

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [6]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [9]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddo

In [10]:
def get_links(url):
    website = Website(url)
   
    response = openai.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)
print(get_links("https://edwarddonner.com"))

{'links': [{'type': 'about page', 'url': 'https://edwardsonner.com/about'}, {'type': 'news', 'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}


In [ ]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

In [ ]:
get_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [14]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [ ]:
print(get_all_details("https://huggingface.co"))

In [11]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'}, {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise/'}, {'type': 'docs', 'url': 'https://huggingface.co/docs'}, {'type': 'changelog', 'url': 'https://www.github.com/huggingface'}]}


"You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nblack-forest-labs/FLUX.1-Kontext-dev\nUpdated\n4 days ago\n•\n110k\n•\n1.09k\ntencent/Hunyuan-A13B-Instruct\nUpdated\nabout 2 hours ago\n•\n5.02k\n•\n642\ngoogle/gemma-3n-E4B-it\nUpdated\n4 days ago\n•\n90k\n•\n331\nOmniGen2/OmniGen2\nUpdated\n7 days ago\n•\n30.8k\n•\n287\ngoogle/magenta-realtime\nUpdated\n8 days ago\n•\n420\nBrowse 1M+ models\nSpaces\nRunning\n8.97k\n8.97k\nDeepSite v2\n🐳

In [20]:
def create_brochure(company_name, url):
    system_prompt = """
You are a professional business content writer and branding expert.
Your task is to analyze and summarize the company-related text provided by the user Your task is to analyze the website content of a company named "{company_name}" , which is typically extracted from a website or company profile.

Based on the content, generate a concise and compelling **company brochure** that includes the following sections if the information is available:
1. Company Overview
2. Mission & Vision
3. Core Products or Services
4. Key Features, Strengths, or Values
5. Contact Information

Make the brochure clear, engaging, and suitable for clients, investors, or partners.
Only use the information from the user input — do not add or assume any external details.
Ensure the language is professional, informative, and marketing-friendly.
"""

    response = openai.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'}, {'type': 'enterprise', 'url': 'https://huggingface.co/endpoints'}, {'type': 'pricing#endpoints', 'url': 'https://huggingface.co/pricing-endpoints'}]}


**Hugging Face – The AI Community Building the Future**

**[Cover Image: Hugging Face Logo and Brand Colors]**

Welcome to Hugging Face, the platform where machine learning enthusiasts come together to collaborate on models, datasets, and applications. We're building a community that enables accelerating innovation in AI through open-source tools, accessible training data, and collaborative development.

### [Company Overview]

* **Mission:** `To build a global community of experts sharing, collaborating, and innovating in the field of artificial intelligence, while advancing its application in various sectors`.
* **Values:`
	+ Openness: Access to public datasets, documentation, and collaboration opportunities.
	+ Excellence: State-of-the-art models, tools, and training data.
	+ Collaboration: Integration with other open-source community platforms.
* **Products/Services:** Models (e.g., transformers, diffusers), Datasets, Spaces for hosting applications and collaborating on them.

### [Mission & Vision]

Our mission is to make AI accessible to everyone worldwide. With a distributed, user-centered approach, we drive progress through our community-driven platform.

```plaintext
"The Future of Artificial Intelligence Starts Here.
With Hugging Face,
We Collaborate.
The World Unites.
"""
```

* **Vision:** `To become the most advanced collaborative AI community for accelerating innovation in various domains.`

### [Core Products/Services]

Our core offerings include:

* **Models:** Leveraging our open-source model hub, we offer multiple architectures and fine-tuning options.
* **Datasets:** Public datasets covering text, image, and video, including institutional data available on GitHub.

```markdown
### Models
#### Transformers
Leverage 146,271 state-of-the-art models like BERT, RoBERTa, and XLNet for various NLP tasks.
```

* **Datasets:** Access unparalleled amounts of datasets in text, image, audio/voice, and 3D modalities.

```markdown
### Datasets
#### Text Generation Inference
Serve language models with optimized TGI toolkit to accelerate training.
```

### [Key Features, Strengths, or Values]

Our strengths lie in:

* **Collaborative Development:** Join our community for contributing to open-source projects and training AI models on diverse datasets.
* **User-Friendly Interface:** Accessible models and APIs enable rapid development with ease.

```markdown
### Key Features
#### Support Community
Join us, expand your knowledge, and contribute to AI innovations in various domains.
```

* **Stronger Together**: `Stay updated`, stay engaged. Learn more about Hugging Face at [HuggingFace.github.io](https://github.com/huggingface).

### [Contact Information]

For professional inquiries or collaborations:

Email: `[info@huggingface.com](mailto:info@huggingface.com)`
Phone: `(XX) XX-XX-XXXX`
Website: `https://huggingfacelab.com`

Don't hesitate to reach out. We hope you'll soon be a part of the Hugging Face community!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://endpoints.huggingface.co'}, {'type': 'hugging-face', 'url': 'https://huggingface.co'}]}


# Hugging Face - The AI Community Building the Future

## Introduction

Hugging Face is an open-source platform that empowers the AI community to build, discover, and collaborate on machine learning models, datasets, and applications. By providing a platform for collaboration and deployment of AI models, we enable organizations to accelerate their AI journey.

## Models

Our platform hosts over 1 million+ models on dedicated, fully managed infrastructure, making it easy to deploy Transformers, Diffusers, or any model. We offer secure, compliant, and flexible production solution that keeps your costs low.

### Catalog

Browse our catalog of hand-picked, ready-to-deploy models across multiple modalities:
* **Text Generation**: accelerated text generation inference with Nvidia GPUs
	+ [Meta-Llama](meta-llama/) / Llama 3.1-70B-Instruct
	+ [Qwen/Twitter](qwen2.5-Coder-7B-Instruct)
	+ ...
### Datasets

Explore access to datasets for any ML task with our 250k+ dataset collection:
* **Images**: datasets for image generation, processing, and classification
	+ (HuggingFaceFW/fineweb-2)
	+ (institutions/institutional-books-1.0)
• ...
### Spaces

Collaborate on unlimited public models, datasets, and applications in our community-driven platform:
* **Video**: datasets for video generation, processing, and inference
	+ (OmniGen2/OmniGen2) / Omnigeneration 2
	+ (Sparc3D/Next-Gen High-Resolution 3D Model Generation)

## Community

Experience the power of AI with our diverse community:
* **Discord**: Join our platform to connect with like-minded individuals and participate in discussions.
* **Twitter**: Stay up-to-date on the latest Hugging Face news, updates, and events.

## Docs

Explore our documentation and guides for a comprehensive introduction to Hugging Face:

### Inference Endpoints

Learn how to deploy Transformers, Diffusers or any model on dedicated, fully managed infrastructure. Keep your costs low with our secure, compliant and flexible production solution.

[Log In](Log In) | [Machine Learning At Your Service](log-in)

## Pricing

Start your AI journey with our scalable pricing plans:

### Compute

Deploy on optimized infrastructure:
* **GPU**: deploy on optimal GPUs for fast inference
* **Inference Endpoints**: manage deployed models and optimize their performance

Starting at $0.60/hour for GPU | Enterprise Plans

### Enterprise

Give your team the most advanced platform to build AI with enterprise-grade security, access controls and dedicated support.

Enroll in our Enterprise plan starting at $20/user/month | Contact us to learn more

## Log In / Sign Up

No Hugging Face account ?
Sign up
![Sign-up Button](Sign-up.png)

Learn More
One-click inference deployment 
Import your favorite model from the Hugging Face Hub or browse our catalog of hand-picked, ready-to-deploy models !

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>